# Hopfield 神经网络实验：二值模式记忆与噪声恢复（学生练习版）

本练习实现一个 **Hopfield 神经网络 Hopfield Neural Network**，用于完成简单二值图像模式的记忆与恢复。

本练习已经保留：

1. Hopfield 网络背景知识。
2. 二值图像模式构造。
3. 原始模式可视化。
4. 噪声与遮挡构造。
5. 恢复过程可视化。
6. 能量曲线绘制。
7. 多模式恢复对比实验。

需要学生补充的关键算法内容：

1. 使用 Hebb 学习规则计算权重矩阵。
2. 计算 Hopfield 网络能量函数。
3. 实现二值符号函数。
4. 实现异步状态更新。
5. 实现同步状态更新。
6. 实现完整迭代恢复流程。

说明：Hopfield 网络不是 BP 前馈神经网络，因此本实验不涉及 BP 中的“前向传播 + 反向传播”训练。它的核心流程是 **存储模式 → 输入受损模式 → 状态迭代更新 → 能量下降 → 收敛恢复**。

## 1. Hopfield 神经网络相关背景知识

### 1.1 联想记忆

Hopfield 网络是一种经典的循环神经网络 Recurrent Neural Network。它常用于模拟 **联想记忆 Associative Memory**。

所谓联想记忆，是指网络在看到一个不完整或带噪声的输入后，能够根据已经记住的模式，逐步恢复到最接近的完整记忆模式。

例如，网络已经记住数字 `0`、`1`、`2`。当输入一个被噪声破坏的 `0` 时，网络通过迭代更新，最终可能恢复出清晰的 `0`。

### 1.2 二值神经元状态

Hopfield 网络中的每个神经元通常只有两个状态：

$$
s_i \in \{-1, +1\}
$$

在二值图像中，可以把：

- `+1` 表示黑色像素或激活状态。
- `-1` 表示白色像素或非激活状态。

如果一张图像大小为 `7 × 7`，那么可以把它展开成一个长度为 `49` 的向量：

$$
s = [s_1, s_2, ..., s_{49}]
$$

Hopfield 网络中的每个神经元对应图像中的一个像素。

### 1.3 权重矩阵与自反馈连接

Hopfield 网络是一个全连接网络。任意两个神经元之间都有连接权重：

$$
w_{ij}
$$

经典 Hopfield 网络要求：

1. 权重矩阵对称：

$$
w_{ij}=w_{ji}
$$

2. 没有自反馈连接：

$$
w_{ii}=0
$$

也就是说，神经元不会直接连接到自己。

### 1.4 Hebb 学习规则

如果要记忆多个二值模式，可以使用 Hebb 学习规则构造权重矩阵。

假设有 `P` 个要存储的模式：

$$
x^{(1)}, x^{(2)}, ..., x^{(P)}
$$

每个模式都是长度为 `N` 的向量，且元素取值为 `-1` 或 `+1`。权重矩阵可以计算为：

$$
W=\frac{1}{N}\sum_{p=1}^{P}x^{(p)}(x^{(p)})^T
$$

之后把主对角线置零：

$$
w_{ii}=0
$$

这样，网络就把这些模式作为稳定状态存储了起来。

### 1.5 状态迭代更新

Hopfield 网络恢复记忆时，会不断更新每个神经元的状态。对于第 `i` 个神经元，输入为：

$$
h_i=\sum_j w_{ij}s_j
$$

然后根据符号函数更新：

$$
s_i =
\begin{cases}
1, & h_i \ge 0 \\
-1, & h_i < 0
\end{cases}
$$

本实验提供两种更新方式：

| 更新方式 | 含义 |
|---|---|
| 同步更新 | 一次性更新所有神经元 |
| 异步更新 | 每次只更新一个神经元或按顺序逐个更新 |

经典 Hopfield 网络更常使用异步更新，因为它更容易保证能量函数单调下降。

### 1.6 能量函数

Hopfield 网络可以定义一个能量函数：

$$
E(s)=-\frac{1}{2}s^TWs
$$

其中：

- `s` 是当前网络状态。
- `W` 是权重矩阵。

在满足权重对称、无自连接，并采用异步更新时，网络状态更新会使能量逐渐下降或保持不变，最终收敛到一个稳定状态。

这个稳定状态可能是存储的原始模式，也可能是某个局部极小点。因此 Hopfield 网络具有“从不完整输入回忆完整模式”的能力，但也可能出现错误恢复。

## 2. 学生任务：完成 Hopfield 网络核心算法

Hopfield 网络没有 BP 神经网络中的反向传播过程。本实验中需要补全的是 Hopfield 网络的核心计算流程。

### 2.1 存储模式应该先做什么、再做什么？

目标：把若干个二值模式存储到权重矩阵 `W` 中。

```text
输入：pattern_vectors，形状为 (P, N)
      P 表示模式数量，N 表示神经元数量

步骤 1：创建 N × N 的全零权重矩阵 W

步骤 2：对每一个存储模式 p：
        计算外积 outer(p, p)
        将结果累加到 W

步骤 3：用神经元数量 N 对 W 做归一化
        W = W / N

步骤 4：去掉自反馈连接
        将 W 的主对角线置为 0

输出：权重矩阵 W
```

### 2.2 能量函数应该怎么计算？

目标：判断当前状态是否逐渐向稳定状态收敛。

```text
输入：当前状态 state，权重矩阵 W

步骤 1：计算 state @ W @ state
步骤 2：乘以 -0.5

输出：能量 E
```

公式：

$$
E(s)=-\frac{1}{2}s^TWs
$$

### 2.3 状态更新应该先做什么、再做什么？

目标：根据其他神经元的状态更新当前神经元。

异步更新流程：

```text
输入：当前状态 state，权重矩阵 W，更新顺序 order

步骤 1：复制当前状态，得到 new_state

步骤 2：按照 order 中的神经元编号逐个更新

步骤 3：对第 i 个神经元：
        net_input = W[i] @ new_state

步骤 4：根据 net_input 的符号更新状态：
        如果 net_input >= 0，则 new_state[i] = +1
        否则 new_state[i] = -1

步骤 5：记录本轮是否发生了状态变化

输出：new_state, changed
```

同步更新流程：

```text
输入：当前状态 state，权重矩阵 W

步骤 1：一次性计算所有神经元输入：
        net_inputs = W @ state

步骤 2：对所有 net_inputs 使用符号函数
        new_state = sign_binary(net_inputs)

步骤 3：判断 new_state 是否与 state 不同

输出：new_state, changed
```

### 2.4 完整恢复过程应该先做什么、再做什么？

目标：从受损图像出发，通过迭代恢复存储模式。

```text
输入：initial_state, W, max_steps

步骤 1：复制 initial_state 作为当前 state
步骤 2：记录初始 state 和初始能量
步骤 3：循环执行状态更新
步骤 4：每更新一轮，就记录新的 state 和 energy
步骤 5：如果本轮没有状态变化，说明已经收敛，可以提前停止

输出：states, energies
```

补全这些 TODO 后，再运行后续噪声恢复、遮挡恢复、状态变化和能量曲线代码。

## 3. 导入基础库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4. 构造二值图像模式

下面构造若干个 `7 × 7` 的简单二值图像模式，包括数字和字母。字符串中的：

- `#` 表示黑色像素，转换为 `+1`
- `.` 表示白色像素，转换为 `-1`

In [ ]:
PATTERN_TEXTS = {
    "0": [
        ".#####.",
        "##...##",
        "##...##",
        "##...##",
        "##...##",
        "##...##",
        ".#####.",
    ],
    "1": [
        "...##..",
        "..###..",
        ".####..",
        "...##..",
        "...##..",
        "...##..",
        ".######",
    ],
    "2": [
        ".#####.",
        "##...##",
        ".....##",
        "...###.",
        "..##...",
        ".##....",
        "#######",
    ],
    "A": [
        "...#...",
        "..###..",
        ".##.##.",
        "##...##",
        "#######",
        "##...##",
        "##...##",
    ],
    "X": [
        "##...##",
        ".##.##.",
        "..###..",
        "...#...",
        "..###..",
        ".##.##.",
        "##...##",
    ],
}


def text_to_pattern(lines):
    """
    将由 # 和 . 组成的文本图案转换为 -1/+1 矩阵。

    参数：
        lines: 字符串列表，每个字符串表示一行图像

    返回：
        pattern: 二值矩阵，# 为 +1，. 为 -1
    """
    array = []
    for line in lines:
        row = [1 if ch == "#" else -1 for ch in line]
        array.append(row)
    return np.array(array, dtype=np.int8)


patterns = {name: text_to_pattern(lines) for name, lines in PATTERN_TEXTS.items()}

for name, pattern in patterns.items():
    print(name, pattern.shape, "取值：", np.unique(pattern))

## 5. 显示原始记忆模式

In [ ]:
def show_patterns(pattern_dict, title="二值图像模式"):
    """
    显示多个二值图像模式。

    参数：
        pattern_dict: {名称: 二值矩阵}
        title: 图标题
    """
    names = list(pattern_dict.keys())
    cols = len(names)
    plt.figure(figsize=(cols * 2, 2.4))
    for i, name in enumerate(names, start=1):
        plt.subplot(1, cols, i)
        plt.imshow(pattern_dict[name], cmap="gray_r", vmin=-1, vmax=1)
        plt.title(name)
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_patterns(patterns, title="准备存储到 Hopfield 网络中的原始模式")

## 6. 向量化与模式相似度

Hopfield 网络使用一维状态向量表示图像。下面定义图像矩阵和向量之间的转换函数。

In [ ]:
def flatten_pattern(pattern):
    """
    将二维二值图像展开为一维向量。

    参数：
        pattern: 二值图像矩阵，形状为 (H, W)

    返回：
        vector: 一维向量，形状为 (H*W,)
    """
    return pattern.reshape(-1).astype(np.int8)


def reshape_pattern(vector, shape=(7, 7)):
    """
    将一维状态向量还原为二维图像。

    参数：
        vector: 一维状态向量
        shape: 图像形状

    返回：
        pattern: 二维二值图像
    """
    return vector.reshape(shape)


def pattern_similarity(a, b):
    """
    计算两个二值模式的像素一致比例。

    参数：
        a: 模式向量或矩阵
        b: 模式向量或矩阵

    返回：
        similarity: 相同位置状态一致的比例
    """
    return np.mean(a.reshape(-1) == b.reshape(-1))


stored_vectors = np.array([flatten_pattern(p) for p in patterns.values()])
stored_names = list(patterns.keys())

print("存储模式矩阵形状：", stored_vectors.shape)

## 7. 实现 Hopfield 网络：请补全 TODO

下面代码已经给出函数名、参数和返回值。请在 `TODO` 位置补全 Hopfield 网络的关键算法。

建议补全顺序：

1. `train_hopfield()`：使用 Hebb 规则计算权重矩阵。
2. `energy()`：计算状态能量。
3. `sign_binary()`：实现二值符号函数。
4. `update_async()`：实现异步更新。
5. `update_sync()`：实现同步更新。
6. `run_hopfield()`：实现完整迭代恢复流程。

In [ ]:
def train_hopfield(pattern_vectors):
    """
    使用 Hebb 学习规则训练 Hopfield 网络。

    参数：
        pattern_vectors: 待存储模式，形状为 (P, N)，元素为 -1 或 +1

    返回：
        W: 权重矩阵，形状为 (N, N)
    """
    # TODO 1：
    # 1. 获取模式数量 num_patterns 和神经元数量 num_neurons
    # 2. 创建形状为 (num_neurons, num_neurons) 的全零矩阵 W
    # 3. 对每个模式 p，累加 np.outer(p, p)
    # 4. 使用 num_neurons 对 W 归一化
    # 5. 使用 np.fill_diagonal(W, 0) 去掉自反馈连接
    # 6. 返回 W
    raise NotImplementedError("请补全 train_hopfield 函数")


def energy(state, W):
    """
    计算 Hopfield 网络能量。

    参数：
        state: 当前状态向量，形状为 (N,)
        W: 权重矩阵，形状为 (N, N)

    返回：
        E: 能量值
    """
    # TODO 2：
    # 根据公式 E = -0.5 * state @ W @ state 计算能量
    raise NotImplementedError("请补全 energy 函数")


def sign_binary(x):
    """
    二值符号函数。x >= 0 返回 +1，否则返回 -1。

    参数：
        x: 输入数值或数组

    返回：
        -1/+1 状态
    """
    # TODO 3：
    # 使用 np.where(x >= 0, 1, -1) 实现符号函数
    raise NotImplementedError("请补全 sign_binary 函数")


def update_async(state, W, order=None):
    """
    对 Hopfield 网络执行一次异步更新。

    参数：
        state: 当前状态向量
        W: 权重矩阵
        order: 神经元更新顺序，None 表示按 0 到 N-1 顺序更新

    返回：
        new_state: 更新后的状态向量
        changed: 本轮是否发生状态变化
    """
    # TODO 4：
    # 1. 复制 state，得到 new_state
    # 2. 如果 order 为 None，则设置为 np.arange(num_neurons)
    # 3. 对 order 中每个神经元 i：
    #    net_input = W[i] @ new_state
    #    如果 net_input >= 0，则 new_state[i] = 1，否则为 -1
    # 4. 如果任意神经元发生变化，则 changed = True
    # 5. 返回 new_state, changed
    raise NotImplementedError("请补全 update_async 函数")


def update_sync(state, W):
    """
    对 Hopfield 网络执行一次同步更新。

    参数：
        state: 当前状态向量
        W: 权重矩阵

    返回：
        new_state: 更新后的状态向量
        changed: 本轮是否发生状态变化
    """
    # TODO 5：
    # 1. 一次性计算所有神经元输入 W @ state
    # 2. 使用 sign_binary 得到 new_state
    # 3. 使用 np.array_equal 判断状态是否发生变化
    # 4. 返回 new_state.astype(np.int8), changed
    raise NotImplementedError("请补全 update_sync 函数")


def run_hopfield(initial_state, W, max_steps=20, mode="async", random_order=True, random_state=42):
    """
    从初始状态开始迭代运行 Hopfield 网络。

    参数：
        initial_state: 初始状态向量
        W: 权重矩阵
        max_steps: 最大迭代次数
        mode: 更新方式，"async" 或 "sync"
        random_order: 异步更新时是否使用随机顺序
        random_state: 随机种子

    返回：
        states: 每一步状态列表
        energies: 每一步能量列表
    """
    # TODO 6：
    # 1. 创建随机数生成器 rng
    # 2. 复制 initial_state 作为当前 state，并转为 np.int8
    # 3. 创建 states 列表，保存初始 state
    # 4. 创建 energies 列表，保存初始能量
    # 5. 在 max_steps 范围内循环：
    #    - 如果 mode == "async"，调用 update_async
    #    - 如果 mode == "sync"，调用 update_sync
    #    - 否则抛出 ValueError
    #    - 保存新的 state 和 energy
    #    - 如果 changed 为 False，提前 break
    # 6. 返回 states, energies
    raise NotImplementedError("请补全 run_hopfield 函数")


# 补全上面的 TODO 后，运行下面三行检查权重矩阵
W = train_hopfield(stored_vectors)

print("权重矩阵形状：", W.shape)
print("权重矩阵是否对称：", np.allclose(W, W.T))
print("主对角线是否为 0：", np.allclose(np.diag(W), 0))

## 8. 测试原始模式是否稳定

如果存储成功，原始模式输入网络后应尽量保持不变，能量也应处于较低位置。

In [ ]:
for name, vector in zip(stored_names, stored_vectors):
    states, energies = run_hopfield(vector, W, max_steps=5, mode="async", random_order=False)
    final_state = states[-1]
    sim = pattern_similarity(vector, final_state)
    print(f"模式 {name}: 迭代步数={len(states)-1}, 与自身相似度={sim:.3f}, 初始能量={energies[0]:.3f}, 最终能量={energies[-1]:.3f}")

## 9. 加入噪声和遮挡

下面定义两种破坏模式的方法：

1. **随机噪声**：随机翻转一定比例的像素。
2. **遮挡**：把图像中某个区域统一置为白色，即 `-1`。

In [ ]:
def add_noise(pattern, noise_ratio=0.25, random_state=42):
    """
    随机翻转二值图像中的部分像素。

    参数：
        pattern: 原始二值图像
        noise_ratio: 翻转像素比例
        random_state: 随机种子

    返回：
        noisy_pattern: 加噪后的图像
    """
    rng = np.random.default_rng(random_state)
    noisy = pattern.copy().reshape(-1)
    num_flip = int(len(noisy) * noise_ratio)
    flip_indices = rng.choice(len(noisy), size=num_flip, replace=False)
    noisy[flip_indices] *= -1
    return noisy.reshape(pattern.shape)


def add_occlusion(pattern, row_slice=slice(2, 5), col_slice=slice(2, 5), fill_value=-1):
    """
    对二值图像添加矩形遮挡。

    参数：
        pattern: 原始二值图像
        row_slice: 遮挡行范围
        col_slice: 遮挡列范围
        fill_value: 遮挡区域填充值，通常为 -1

    返回：
        occluded_pattern: 遮挡后的图像
    """
    occluded = pattern.copy()
    occluded[row_slice, col_slice] = fill_value
    return occluded


target_name = "A"
original_pattern = patterns[target_name]
noisy_pattern = add_noise(original_pattern, noise_ratio=0.28, random_state=5)
occluded_pattern = add_occlusion(original_pattern, row_slice=slice(2, 5), col_slice=slice(2, 5))

show_patterns(
    {
        "原始 A": original_pattern,
        "噪声 A": noisy_pattern,
        "遮挡 A": occluded_pattern,
    },
    title="原始模式与受损模式"
)

## 10. 使用 Hopfield 网络恢复噪声模式

将带噪声的模式输入网络，观察状态是否逐步恢复到原始模式。

In [ ]:
initial_state = flatten_pattern(noisy_pattern)
states, energies = run_hopfield(
    initial_state,
    W,
    max_steps=20,
    mode="async",
    random_order=True,
    random_state=RANDOM_STATE,
)

final_pattern = reshape_pattern(states[-1])

print("迭代步数：", len(states) - 1)
print("受损模式与原始模式相似度：", pattern_similarity(noisy_pattern, original_pattern))
print("恢复结果与原始模式相似度：", pattern_similarity(final_pattern, original_pattern))

show_patterns(
    {
        "原始模式": original_pattern,
        "输入噪声": noisy_pattern,
        "恢复结果": final_pattern,
    },
    title="Hopfield 网络噪声恢复结果"
)

## 11. 可视化状态迭代变化

下面显示网络在恢复过程中的若干中间状态。

In [ ]:
def show_state_sequence(states, shape=(7, 7), max_show=8, title="状态迭代变化"):
    """
    显示 Hopfield 网络状态变化序列。

    参数：
        states: 状态向量列表
        shape: 图像形状
        max_show: 最多显示多少步
        title: 图标题
    """
    show_count = min(len(states), max_show)
    indices = np.linspace(0, len(states) - 1, show_count, dtype=int)

    plt.figure(figsize=(show_count * 1.6, 2.3))
    for plot_i, state_i in enumerate(indices, start=1):
        plt.subplot(1, show_count, plot_i)
        plt.imshow(reshape_pattern(states[state_i], shape), cmap="gray_r", vmin=-1, vmax=1)
        plt.title(f"step {state_i}")
        plt.axis("off")
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


show_state_sequence(states, title="从噪声 A 恢复的状态变化")

## 12. 能量函数下降过程

Hopfield 网络的状态更新会使能量逐渐下降或保持不变。下面绘制恢复过程中的能量变化曲线。

In [ ]:
def plot_energy(energies, title="Hopfield 网络能量变化"):
    """
    绘制能量函数变化曲线。

    参数：
        energies: 能量值列表
        title: 图标题
    """
    plt.figure(figsize=(7, 4))
    plt.plot(range(len(energies)), energies, marker="o", color="#2A6FBB")
    plt.xlabel("迭代步数")
    plt.ylabel("能量 E")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.show()


plot_energy(energies, title="噪声模式恢复过程中的能量下降")

for i, e in enumerate(energies):
    print(f"step {i:02d}: E = {e:.4f}")

## 13. 遮挡模式恢复实验

下面把遮挡后的模式输入 Hopfield 网络，观察恢复效果。

In [ ]:
initial_occluded_state = flatten_pattern(occluded_pattern)
states_occ, energies_occ = run_hopfield(
    initial_occluded_state,
    W,
    max_steps=20,
    mode="async",
    random_order=True,
    random_state=7,
)

final_occluded_pattern = reshape_pattern(states_occ[-1])

print("迭代步数：", len(states_occ) - 1)
print("遮挡模式与原始模式相似度：", pattern_similarity(occluded_pattern, original_pattern))
print("恢复结果与原始模式相似度：", pattern_similarity(final_occluded_pattern, original_pattern))

show_patterns(
    {
        "原始模式": original_pattern,
        "输入遮挡": occluded_pattern,
        "恢复结果": final_occluded_pattern,
    },
    title="Hopfield 网络遮挡恢复结果"
)

show_state_sequence(states_occ, title="从遮挡 A 恢复的状态变化")
plot_energy(energies_occ, title="遮挡模式恢复过程中的能量下降")

## 14. 多模式恢复对比实验

下面对多个已存储模式分别加入噪声，并观察恢复结果。

In [ ]:
def recover_and_summarize(pattern_name, noise_ratio=0.22, seed=0):
    """
    对指定模式加噪并使用 Hopfield 网络恢复。

    参数：
        pattern_name: 模式名称
        noise_ratio: 噪声比例
        seed: 随机种子

    返回：
        result: 包含原始、噪声、恢复模式和相似度的字典
    """
    original = patterns[pattern_name]
    noisy = add_noise(original, noise_ratio=noise_ratio, random_state=seed)
    states_i, energies_i = run_hopfield(flatten_pattern(noisy), W, max_steps=20, mode="async", random_state=seed)
    recovered = reshape_pattern(states_i[-1])
    return {
        "name": pattern_name,
        "original": original,
        "noisy": noisy,
        "recovered": recovered,
        "before_similarity": pattern_similarity(noisy, original),
        "after_similarity": pattern_similarity(recovered, original),
        "steps": len(states_i) - 1,
        "energies": energies_i,
    }


results = [recover_and_summarize(name, noise_ratio=0.22, seed=i + 10) for i, name in enumerate(stored_names)]

plt.figure(figsize=(9, len(results) * 2.0))
for row, result in enumerate(results):
    images = [result["original"], result["noisy"], result["recovered"]]
    titles = [
        f"{result['name']} 原始",
        f"加噪 sim={result['before_similarity']:.2f}",
        f"恢复 sim={result['after_similarity']:.2f}",
    ]
    for col, (img, t) in enumerate(zip(images, titles)):
        ax = plt.subplot(len(results), 3, row * 3 + col + 1)
        ax.imshow(img, cmap="gray_r", vmin=-1, vmax=1)
        ax.set_title(t)
        ax.axis("off")

plt.suptitle("不同存储模式的噪声恢复对比", fontsize=14)
plt.tight_layout()
plt.show()

for result in results:
    print(
        f"模式 {result['name']}: "
        f"恢复前相似度={result['before_similarity']:.3f}, "
        f"恢复后相似度={result['after_similarity']:.3f}, "
        f"迭代步数={result['steps']}"
    )

## 15. 噪声强度对恢复效果的影响

噪声越强，初始状态距离原始记忆越远，网络越可能恢复失败或收敛到其他稳定状态。

In [ ]:
noise_levels = [0.05, 0.10, 0.20, 0.30, 0.40]
target_name = "2"
target_pattern = patterns[target_name]
summary = []

for i, noise_level in enumerate(noise_levels):
    noisy = add_noise(target_pattern, noise_ratio=noise_level, random_state=100 + i)
    states_i, energies_i = run_hopfield(flatten_pattern(noisy), W, max_steps=20, mode="async", random_state=200 + i)
    recovered = reshape_pattern(states_i[-1])
    summary.append((noise_level, noisy, recovered, pattern_similarity(noisy, target_pattern), pattern_similarity(recovered, target_pattern)))

plt.figure(figsize=(len(noise_levels) * 2.2, 4.8))
for i, (noise_level, noisy, recovered, before_sim, after_sim) in enumerate(summary):
    plt.subplot(2, len(noise_levels), i + 1)
    plt.imshow(noisy, cmap="gray_r", vmin=-1, vmax=1)
    plt.title(f"噪声 {noise_level:.0%}\nsim={before_sim:.2f}")
    plt.axis("off")

    plt.subplot(2, len(noise_levels), len(noise_levels) + i + 1)
    plt.imshow(recovered, cmap="gray_r", vmin=-1, vmax=1)
    plt.title(f"恢复\nsim={after_sim:.2f}")
    plt.axis("off")

plt.suptitle("噪声强度对 Hopfield 恢复效果的影响", fontsize=14)
plt.tight_layout()
plt.show()

for noise_level, _, _, before_sim, after_sim in summary:
    print(f"噪声比例={noise_level:.0%}, 恢复前相似度={before_sim:.3f}, 恢复后相似度={after_sim:.3f}")

## 16. 实验拓展

可以尝试修改以下内容：

1. 增加或减少存储模式数量，观察恢复效果变化。
2. 修改图案大小，例如从 `7 × 7` 扩展到 `9 × 9`。
3. 对比同步更新和异步更新的差异。
4. 增大噪声比例，观察网络恢复失败的情况。
5. 构造相似度更高的模式，观察是否更容易互相混淆。

思考问题：

1. 为什么 Hopfield 网络要求权重矩阵对称？
2. 为什么主对角线要置零？
3. 能量函数下降说明了什么？
4. 当存储模式太多时，为什么恢复效果会变差？
5. Hopfield 网络为什么可以看作一种联想记忆模型？

## 17. 实验小结

本实验从零实现了一个 Hopfield 神经网络，并完成了二值图像模式的记忆与噪声恢复。

通过实验可以看到：

1. Hopfield 网络能够把若干二值模式存储为稳定状态。
2. 带噪声或遮挡的输入可以通过状态迭代逐渐恢复。
3. 异步更新过程中，能量函数通常会下降或保持不变。
4. 当噪声过强或存储模式过多时，网络可能收敛到错误模式或局部稳定状态。
5. Hopfield 网络体现了“联想记忆、自反馈连接、状态迭代更新、能量函数收敛”等重要思想。